In [ ]:
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch

from ugdatalab.utils.compose import Compose
from ugdatalab.models.galaxy_zoo import GalaxyZooDataset
from ugdatalab.models.galaxy_zoo.constants import N_LABELS
from ugdatalab.methods.neural_network.cnn import train_cnn, count_parameters
from ugdatalab.methods.neural_network.augmentation import CenterCrop

from architectures import build_custom_cnn
import plotters

# Galaxy Image Classification — Custom CNN

## Task 16 — Custom CNN Architecture

We build a custom CNN following the lab manual guidelines and the lecture's CNN template: convolution layers with batch normalization and pooling, followed by fully connected layers with dropout, ending in a sigmoid activation to constrain outputs to $[0, 1]$.

**Architecture choices:**
- **4 convolutional blocks** (32 → 64 → 128 → 256 channels), each `Conv2d → BatchNorm2d → ReLU → MaxPool2d`. The increasing channel depth allows the network to learn progressively more abstract features: edges → textures → morphological components → galaxy-scale structure. Adding the 4th block reduces the spatial map from 8×8 to 4×4, shrinking the FC input by 4× and shifting capacity into the convolutional stack. Batch normalization stabilizes training and acts as a mild regularizer.
- **3×3 kernels throughout**, matching the lecture's CNN convention. Stacking 3×3 convs is the standard choice once spatial dimensions are reduced and keeps the parameter count modest.
- **2 fully connected layers** (256, 128 units) with ReLU activation and 50% dropout. Dropout is the primary regularization mechanism for the FC head, preventing it from memorizing training examples.
- **Adam optimizer** with initial learning rate $10^{-3}$.
- **Sigmoid output** ensures all 37 predictions lie in $[0, 1]$, matching the label range (multi-label regression rather than the softmax classification shown in lecture).

Target: RMSE $\leq 0.11$.

In [ ]:
# Load preprocessed data
img_data = np.load("artifacts/galaxy_zoo_images.npz")
images = img_data["images"]
label_data = np.load("artifacts/galaxy_zoo_labels.npz")
labels = label_data["labels"]
split_data = np.load("artifacts/split_indices.npz")
train_idx, val_idx = split_data["train_idx"], split_data["val_idx"]

train_images, val_images = images[train_idx], images[val_idx]
train_labels, val_labels = labels[train_idx], labels[val_idx]
CACHE_SIZE = images.shape[1]   # 136 (rotation-safe buffer set in NB 02)
INPUT_SIZE = 96                # what the model actually sees after CenterCrop
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

default_transform = Compose([CenterCrop(INPUT_SIZE)])
train_ds = GalaxyZooDataset(train_images, train_labels, transform=default_transform)
val_ds = GalaxyZooDataset(val_images, val_labels, transform=default_transform)

In [ ]:
model = build_custom_cnn(
    n_labels=N_LABELS,
    n_channels_list=[32, 64, 128, 256],
    kernel_sizes=[3, 3, 3, 3],
    fc_sizes=[256, 128],
    dropout_rate=0.5,
    pool_type="max",
    input_size=INPUT_SIZE,
)
print(model)
print(f"\nCustom CNN trainable parameters: {count_parameters(model):,}")

In [ ]:
_ckpt_pt = Path("artifacts/custom_cnn.pt")
_ckpt_npz = Path("artifacts/custom_result.npz")
if _ckpt_pt.exists() and _ckpt_npz.exists():
    _data = np.load(_ckpt_npz)
    custom_result = SimpleNamespace(
        model_state=torch.load(_ckpt_pt, map_location=DEVICE),
        train_losses=_data["train_losses"],
        val_losses=_data["val_losses"],
        best_epoch=int(_data["best_epoch"]),
        best_val_loss=float(_data["best_val_loss"]),
        n_parameters=int(_data["n_parameters"]),
        learning_rates=_data["learning_rates"],
    )
    print(f"Loaded cached artifacts/custom_cnn.pt (best val RMSE: {custom_result.best_val_loss:.4f})")
else:
    custom_result = train_cnn(
        model=model,
        train_dataset=train_ds,
        val_dataset=val_ds,
        batch_size=768,
        n_epochs=100,
        lr=1e-3,
        seed=42,
        optimizer_factory=lambda params, lr: torch.optim.Adam(params, lr=lr),
        scheduler_factory=None,
        num_workers=0,
    )
    print(f"\nBest epoch: {custom_result.best_epoch + 1}")
    print(f"Best validation RMSE: {custom_result.best_val_loss:.4f}")

In [ ]:
ax = plotters.plot_loss_curves(
    custom_result.train_losses, custom_result.val_losses, "Custom CNN",
)
plt.show()

## Task 18 — Parameter Count Comparison

We compare the number of trainable parameters in our custom CNN vs. ResNet-18, and compare both to the total number of pixels in the compressed training set. If the number of parameters greatly exceeds the number of training pixels, the model has more capacity than the data can constrain, which should make us cautious about overfitting.

In [ ]:
import pandas as pd

resnet_data = np.load("artifacts/resnet_result.npz")
resnet_params = int(resnet_data["n_parameters"])
custom_params = custom_result.n_parameters
total_pixels = train_images.shape[0] * INPUT_SIZE * INPUT_SIZE * 3

comparison = pd.DataFrame({
    "Model": ["Custom CNN", "ResNet-18", "Training pixels"],
    "Count": [f"{custom_params:,}", f"{resnet_params:,}", f"{total_pixels:,}"],
})
print(comparison.to_string(index=False))
print(f"\nCustom CNN / training pixels ratio: {custom_params / total_pixels:.2f}")
print(f"ResNet-18 / training pixels ratio: {resnet_params / total_pixels:.2f}")

## Task 19 — Model Selection

We compare the best validation RMSE from both models and select the better-performing one for optimization in NB 05.

In [ ]:
resnet_best = float(resnet_data["best_val_loss"])
custom_best = custom_result.best_val_loss

print(f"ResNet-18 best val RMSE: {resnet_best:.4f}")
print(f"Custom CNN best val RMSE: {custom_best:.4f}")

best_model = "resnet" if resnet_best < custom_best else "custom"
print(f"\nBest model: {'ResNet-18' if best_model == 'resnet' else 'Custom CNN'}")

# Architecture hyperparameters used to build the custom CNN (read by NB 05/06)
custom_n_channels_list = [32, 64, 128, 256]
custom_kernel_sizes = [3, 3, 3, 3]
custom_fc_sizes = [256, 128]
custom_dropout_rate = 0.5
custom_pool_type = "max"

# Save custom CNN results
torch.save(custom_result.model_state, "artifacts/custom_cnn.pt")
np.savez_compressed(
    "artifacts/custom_result.npz",
    train_losses=custom_result.train_losses,
    val_losses=custom_result.val_losses,
    best_epoch=custom_result.best_epoch,
    best_val_loss=custom_result.best_val_loss,
    n_parameters=custom_result.n_parameters,
    learning_rates=custom_result.learning_rates,
    best_model=best_model,
    n_channels_list=np.array(custom_n_channels_list),
    kernel_sizes=np.array(custom_kernel_sizes),
    fc_sizes=np.array(custom_fc_sizes),
    dropout_rate=custom_dropout_rate,
    pool_type=custom_pool_type,
)
print("Saved artifacts/custom_cnn.pt and artifacts/custom_result.npz")